# Multi Attack Runner
Coordinate PWSCUP2025 attack scripts with reusable helpers and grouping.


In [1]:
import os
import sys
import subprocess
from pathlib import Path
from shutil import which
from typing import Dict, Optional, Sequence


In [2]:
## Default team IDs (1-24 except 21)
TEAM_IDS = tuple(i for i in range(1, 25) if i != 21)
# TEAM_IDS = (22, )

def get_teams():
    '''Return the default sequence of team IDs.'''
    return TEAM_IDS


In [3]:

def loop_for_all_teams(
    command_template,
    *,
    teams=None,
    dry_run=False,
    strict=True,
    continue_on_error=False,
    cwd=None
):
    '''
    command_template: ['python', 'attack/attack_Ci.py', '...{id:02d}...', ...]
    teams: explicit iterable of team ids. Defaults to get_teams().
    dry_run: True -> only print expanded commands.
    strict: True -> require at least one {id:02d} placeholder.
    continue_on_error: True -> keep going after failures.
    cwd: working directory for subprocess.run.
    '''
    id_indices = [
        i for i, arg in enumerate(command_template)
        if isinstance(arg, str) and "{id:02d}" in arg
    ]
    if strict and not id_indices:
        raise ValueError(f"No {{id:02d}} placeholder found in: {command_template}")

    cmd0 = command_template[:]
    if cmd0 and cmd0[0] in ("python", "python3"):
        cmd0[0] = sys.executable

    exe = cmd0[0]
    if os.path.sep not in exe and which(exe) is None:
        raise RuntimeError(f"Executable not found on PATH: {exe}")

    team_list = get_teams() if teams is None else teams

    for team in team_list:
        cmd = cmd0[:]
        for ind in id_indices:
            cmd[ind] = cmd[ind].format(id=team)

        def is_out_flag(i: int) -> bool:
            if i == 0:
                return False
            prev = cmd[i - 1]
            return isinstance(prev, str) and (
                prev in ("-o", "--out", "--out-map", "--out-pred", "--out-conf")
                or prev.startswith("--out")
            )

        missing_inputs = []
        for i, arg in enumerate(cmd):
            if isinstance(arg, str) and arg.lower().endswith((".csv", ".json")) and not is_out_flag(i):
                candidate = arg if cwd is None else os.path.join(cwd, arg)
                if not os.path.exists(candidate):
                    missing_inputs.append(arg)

        print(">>", " ".join(cmd))
        if missing_inputs:
            message = f"[team {team}] Missing input files: {missing_inputs}"
            if continue_on_error:
                print("!!", message)
                continue
            raise FileNotFoundError(message)

        if dry_run:
            continue

        try:
            completed = subprocess.run(
                cmd,
                check=True,
                cwd=cwd,
                capture_output=True,
                text=True,
                encoding="utf-8",
                errors="replace",
            )
            if completed.stdout:
                print(completed.stdout.strip())
        except subprocess.CalledProcessError as exc:
            print(f"[ERROR] team {team} command failed with code {exc.returncode}")
            if exc.stdout:
                print("--- stdout ---")
                print(exc.stdout.strip())
            if exc.stderr:
                print("--- stderr ---")
                print(exc.stderr.strip())
            if not continue_on_error:
                raise


In [4]:

COMMAND_REGISTRY: Dict[str, Dict[str, object]] = {}
PIPELINE_ORDER: list[str] = []


def register_command(
    key: str,
    *,
    label: str,
    template: Sequence[str],
    category: str,
    description: str = "",
    cwd: Optional[str] = None,
    continue_on_error: bool = False,
) -> None:
    COMMAND_REGISTRY[key] = {
        "label": label,
        "template": list(template),
        "category": category,
        "description": description,
        "cwd": cwd,
        "continue_on_error": continue_on_error,
    }
    if key not in PIPELINE_ORDER:
        PIPELINE_ORDER.append(key)


def run_command(
    key: str,
    *,
    label: str,
    template: Sequence[str],
    category: str,
    description: str = "",
    teams: Optional[Sequence[int]] = None,
    dry_run: bool = False,
    continue_on_error: bool = False,
    cwd: Optional[str] = None,
) -> None:
    register_command(
        key,
        label=label,
        template=template,
        category=category,
        description=description,
        cwd=cwd,
        continue_on_error=continue_on_error,
    )
    print(f"=== {label} ({key}) ===")
    loop_for_all_teams(
        template,
        teams=teams,
        dry_run=dry_run,
        continue_on_error=continue_on_error,
        cwd=cwd,
    )


def rerun_command(
    key: str,
    *,
    teams: Optional[Sequence[int]] = None,
    dry_run: bool = False,
    continue_on_error: Optional[bool] = None,
    cwd: Optional[str] = None,
) -> None:
    if key not in COMMAND_REGISTRY:
        raise KeyError(f"Unknown command key: {key}")
    spec = COMMAND_REGISTRY[key]
    effective_continue = (
        spec.get("continue_on_error", False)
        if continue_on_error is None
        else continue_on_error
    )
    print(f"=== {spec['label']} ({key}) ===")
    loop_for_all_teams(
        spec["template"],
        teams=teams,
        dry_run=dry_run,
        continue_on_error=effective_continue,
        cwd=cwd or spec.get("cwd"),
    )


def list_commands(category: Optional[str] = None) -> None:
    for key in PIPELINE_ORDER:
        spec = COMMAND_REGISTRY.get(key)
        if spec is None:
            continue
        if category and spec["category"] != category:
            continue
        line = f"{key:>24}  {spec['label']} [{spec['category']}]"
        print(line)
        desc = spec.get("description")
        if desc:
            print(f"    {desc}")


def run_pipeline(
    keys: Optional[Sequence[str]] = None,
    *,
    teams: Optional[Sequence[int]] = None,
    dry_run: bool = False,
    continue_on_error: Optional[bool] = None,
) -> None:
    sequence = keys if keys is not None else PIPELINE_ORDER
    for key in sequence:
        rerun_command(
            key,
            teams=teams,
            dry_run=dry_run,
            continue_on_error=continue_on_error,
        )


TEAM_22_ONLY = (22,)


In [5]:

def find_project_root(start: Path) -> Path:
    markers = ("requirements.txt", "GUIDE_FOR_BEGINNERS.md")
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
print(f"Working directory set to: {PROJECT_ROOT}")


Working directory set to: /home/kikuchih/pwscup2025-scripts


## Mode Configuration
Adjust dataset prefixes for prep or contest runs using the constants below.


In [6]:

MODE_PRESETS = {
    "prep": {
        "all": "A",
        "original": "B",
        "anon": "C",
        "model": "D",
        "variants": ["3"],
        "attack_data": "PWSCUP2025_Pre_Data_for_Attack"
    },
    "contest": {
        "all": "AA",
        "original": "BB",
        "anon": "CC",
        "model": "DD",
        "variants": ["1", "2", "3"],
        "attack_data": "PWSCUP2025_Main_Data_for_Attack"
    },
}

mode = "contest"  # change to "contest" for contest datasets
MODE_CONFIG = MODE_PRESETS[mode]
mode_all = MODE_CONFIG["all"]
mode_original = MODE_CONFIG["original"]
mode_anon = MODE_CONFIG["anon"]
mode_model = MODE_CONFIG["model"]
mode_variants = MODE_CONFIG["variants"]
mode_attack_data = MODE_CONFIG["attack_data"]


## Preprocessing


In [ ]:
# run_command(
#     key="fix_csv",
#     label="Fix anonymized CSV files",
#     category="preprocess",
#     description="Check column ranges and emit *_fix CSV files.",
#     template=[
#         "python",
#         "util/check_and_fix_csv.py",
#         f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
#         "data/pre_columns_range.json",
#         f"in/{mode_attack_data}/{mode_anon}{{id:02d}}_fix.csv",
#     ],
# )


=== Fix anonymized CSV files (fix_csv) ===
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe util/check_and_fix_csv.py in/PWSCUP2025_Main_Data_for_Attack/CC01.csv data/pre_columns_range.json in/PWSCUP2025_Main_Data_for_Attack/CC01_fix.csv


FileNotFoundError: [team 1] Missing input files: ['in/PWSCUP2025_Main_Data_for_Attack/CC01_fix.csv']

## Ci Attacks


### Ci attack (original)

Uses `attack/attack_Ci.py` (baseline `AttackCiNN`) to label Ai rows that become the closest neighbour of any Ci record, writing a 0/1 membership vector to `out_attack/{mode_anon}{id:02d}_inferred.csv`.  The script builds feature matrices with `mia.build_feature_matrices`, runs 1-NN in Manhattan space, and saves hits so downstream mixers can score raw inclusion candidates.

このセルでは `attack/attack_Ci.py` を実行し、Ci の各行に最も近い Ai 行を 0/1 のメンバー候補として `out_attack/{mode_anon}{id:02d}_inferred.csv` に書き出します。スクリプト内で `mia.build_feature_matrices` を使って特徴量を整備し、マンハッタン距離の 1-NN により命中した Ai 行だけを保存します。


In [24]:

run_command(
    key="ci_original",
    label="Ci attack (original)",
    category="ci",
    description="Baseline Ci attack using original samples.",
    template=[
        "python",
        "attack/attack_Ci.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred.csv",
    ],
)


=== Ci attack (original) (ci_original) ===
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci.py in/PWSCUP2025_Main_Data_for_Attack/AA01.csv in/PWSCUP2025_Main_Data_for_Attack/CC01.csv -o out_attack/CC01_inferred.csv
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci.py in/PWSCUP2025_Main_Data_for_Attack/AA02.csv in/PWSCUP2025_Main_Data_for_Attack/CC02.csv -o out_attack/CC02_inferred.csv
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci.py in/PWSCUP2025_Main_Data_for_Attack/AA03.csv in/PWSCUP2025_Main_Data_for_Attack/CC03.csv -o out_attack/CC03_inferred.csv
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci.py in/PWSCUP2025_Main_Data_for_Attack/AA04.csv in/PWSCUP2025_Main_Data_for_Attack/CC04.csv -o out_attack/CC04_inferred.csv
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\p

### Ci attack (extended)

Runs `attack/attack_Ci_ex.py` with `k=1`, producing two columns (`knn_hits`, `min_dist`) for each Ai row.  This extended output preserves the neighbour count and Manhattan distance statistics, enabling later scoring strategies that reward consistently selected candidates while still using the single-neighbour configuration captured in the template.

このセルでは `attack/attack_Ci_ex.py` を `k=1` で呼び出し、Ai 各行について `knn_hits` と `min_dist` の 2 列を計算して `out_attack/{mode_anon}{id:02d}_inferred_ex.csv` に書き込みます。単一近傍を維持しつつ、命中回数と最小距離の統計を残すことで後段のスコアリングに活用できます。


In [25]:

run_command(
    key="ci_extended",
    label="Ci attack (extended)",
    category="ci",
    description="Extended Ci attack with single-neighbour output.",
    template=[
        "python",
        "attack/attack_Ci_ex.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_ex.csv",
        "-k",
        "1",
    ],
)


=== Ci attack (extended) (ci_extended) ===
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci_ex.py in/PWSCUP2025_Main_Data_for_Attack/AA01.csv in/PWSCUP2025_Main_Data_for_Attack/CC01.csv -o out_attack/CC01_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci_ex.py in/PWSCUP2025_Main_Data_for_Attack/AA02.csv in/PWSCUP2025_Main_Data_for_Attack/CC02.csv -o out_attack/CC02_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci_ex.py in/PWSCUP2025_Main_Data_for_Attack/AA03.csv in/PWSCUP2025_Main_Data_for_Attack/CC03.csv -o out_attack/CC03_inferred_ex.csv -k 1
inferred was successfully saved.
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci_ex.py in/PWSCUP2025_Main_Data_for_Attack/AA04.csv in/PWSCUP2025_Main_Data_for_Attack/CC04.csv -o out_attack/CC04_inferred_ex.csv -k 1
inferred was successfully saved.
>

### Ci attack (k-NN)

Invokes `attack/attack_Ci_ex_greedy.py` in `nn` mode with `k=5`.  The script fits a k-NN search on Ci features, counts how many times each Ai row appears across the five-neighbour sets, and records corresponding minimum distances into `out_attack/{mode_anon}{id:02d}_inferred_ex_greedy_k5_nn.csv` for richer ensemble cues.

このセルでは `attack/attack_Ci_ex_greedy.py` を `nn` モード・`k=5` で実行し、Ai 各行が近傍 Ci に選ばれた回数と最小距離を `out_attack/{mode_anon}{id:02d}_inferred_ex_greedy_k5_nn.csv` に保存します。Ci で構築した特徴空間に対して 5 近傍探索を繰り返し、エンサンブル向けの詳細統計を得ています。


In [26]:

run_command(
    key="ci_knn",
    label="Ci attack (k-NN)",
    category="ci",
    description="k-nearest Ci attack (mode=nn, k=5).",
    template=[
        "python",
        "attack/attack_Ci_ex_greedy.py",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        "-m",
        "nn",
        "-k",
        "5",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_ex_greedy_k5_nn.csv",
    ],
)


=== Ci attack (k-NN) (ci_knn) ===
>> c:\Users\kikus\anaconda3\envs\pwscup2025\python.exe attack/attack_Ci_ex_greedy.py in/PWSCUP2025_Main_Data_for_Attack/CC01.csv in/PWSCUP2025_Main_Data_for_Attack/AA01.csv -m nn -k 5 -o out_attack/CC01_inferred_ex_greedy_k5_nn.csv
dists: [[0.24282704 0.33439457 0.34678686 0.34785798 0.36006168]
 [0.29664594 0.36257672 0.38015795 0.43923289 0.52094108]
 [0.24735704 0.24762407 0.25613528 0.27312425 0.29955515]
 ...
 [0.19596189 0.21180037 0.22924529 0.23846032 0.2407373 ]
 [0.1413632  0.14497054 0.16654584 0.18267353 0.19240761]
 [0.17572206 0.21171437 0.21336219 0.21976428 0.22281277]]
inds: [[29383 81900 32114 23906 76352]
 [82678 47740 36095 57616 58706]
 [27926 87324  9931 19430 31558]
 ...
 [94506 14730 59414 55927 54277]
 [73421 48501 35853 61756 25993]
 [63645 88079 19304 32439 16598]]
dists.shape: (10000, 5), inds.shape: (10000, 5)
0.24282704293727875 29383
idx: [29383 81900 32114 ... 19304 32439 16598]
distances: [0.24282704 0.33439457 0.346786

### Ci attack (greedy)

Calls `attack/attack_Ci_ex_greedy.py` in `greedy` mode with `k=300` to approximate a near one-to-one Ci?Ai matching.  The command writes the selected membership vector to `out_attack/{mode_anon}{id:02d}_inferred_ex_greedy_k300_greedy.csv` and, when requested, emits a match map (`ci_idx`, `ai_idx`, `distance`, `rank`) to `out_attack/{mode_anon}{id:02d}_matchmap_k300.csv` for audit.

このセルでは `attack/attack_Ci_ex_greedy.py` を `greedy` モード・`k=300` で実行し、Ci と Ai の準 1 対 1 マッチングを構築します。選ばれたメンバー判定は `out_attack/{mode_anon}{id:02d}_inferred_ex_greedy_k300_greedy.csv` に、対応マップは `out_attack/{mode_anon}{id:02d}_matchmap_k300.csv` に出力され、割り当て内容を後から検証できます。


In [ ]:

run_command(
    key="ci_greedy",
    label="Ci attack (greedy)",
    category="ci",
    description="Greedy Ci attack with k=300 for exhaustive matching.",
    template=[
        "python",
        "attack/attack_Ci_ex_greedy.py",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        "-m",
        "greedy",
        "-k",
        "300",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_ex_greedy_k300_greedy.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_k300.csv",
    ],
)


=== Ci attack (greedy) (ci_greedy) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_Ci_ex_greedy.py in/PWSCUP2025_Pre_Data_for_Attack/C22_fix.csv in/PWSCUP2025_Pre_Data_for_Attack/A22.csv -m greedy -k 300 -o out_attack/C22_inferred_ex_greedy_k300_greedy.csv --out-map out_attack/C22_matchmap_k300.csv
inferred was successfully saved.
match table was successfully saved to out_attack/C22_matchmap_k300.csv
[stats greedy] selected=10000/100000


## Di Attacks


In [ ]:

# run_command(
#     key="di_original",
#     label="Di attack (original)",
#     category="di",
#     description="Baseline Di attack using provided model predictions.",
#     template=[
#         "python",
#         "attack/attack_Di.py",
#         f"in/{mode_attack_data}/{mode_model}{{id:02d}}.json",
#         f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
#     ],
# )


### Di attack (extended)

Executes `attack/attack_Di_ex.py` on the per-team XGBoost booster (`in/{mode_attack_data}/{mode_model}{id:02d}.json`) to export both Pred and Conf membership files.  The script reconstructs features via `analysis/xgbt_train.build_X`, aligns columns with the booster, applies optional thresholds/top-k rules, and saves results to `out_attack/inferred_membership1_{id:02d}_ex.csv` (Pred) and `out_attack/inferred_membership2_{id:02d}_ex.csv` (Conf).

このセルでは `attack/attack_Di_ex.py` を用い、チームごとの XGBoost モデル (`in/{mode_attack_data}/{mode_model}{id:02d}.json`) を読み込んで Pred/Conf のメンバー推定を計算します。`analysis/xgbt_train.build_X` で特徴量を再構築し、しきい値や Top-K 指定を反映した結果を `out_attack/inferred_membership1_{id:02d}_ex.csv` と `out_attack/inferred_membership2_{id:02d}_ex.csv` に保存します。


In [ ]:

run_command(
    key="di_extended",
    label="Di attack (extended)",
    category="di",
    description="Extended Di attack with prediction and confidence outputs.",
    template=[
        "python",
        "attack/attack_Di_ex.py",
        f"in/{mode_attack_data}/{mode_model}{{id:02d}}.json",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        "--out-pred",
        "out_attack/inferred_membership1_{id:02d}_ex.csv",
        "--out-conf",
        "out_attack/inferred_membership2_{id:02d}_ex.csv",
    ],
)


### Di attack (extended, DALL model)

Runs the same `attack/attack_Di_ex.py` pipeline but swaps in the anonymized `out_anonymized/DALL_multi.json` booster so the public DALL model scores the Ai tables.  Outputs mirror the baseline (`..._ex_DALL.csv`) for Pred and Conf, letting downstream combiners compare anonymized model behaviour.

このセルでは 匿名化済みモデル `out_anonymized/DALL_multi.json` を `attack/attack_Di_ex.py` に渡し、公開 Ai テーブルに対する Pred/Conf を `out_attack/inferred_membership1_{id:02d}_ex_DALL.csv` と `out_attack/inferred_membership2_{id:02d}_ex_DALL.csv` に出力します。ベースラインと同じ手順で匿名化モデルの挙動を比較できます。


In [ ]:
# DALL
run_command(
    key="di_extended",
    label="Di attack (extended)",
    category="di",
    description="Extended Di attack with prediction and confidence outputs.",
    template=[
        "python",
        "attack/attack_Di_ex.py",
        f"out_anonymized/DALL_multi.json",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        "--out-pred",
        "out_attack/inferred_membership1_{id:02d}_ex_DALL.csv",
        "--out-conf",
        "out_attack/inferred_membership2_{id:02d}_ex_DALL.csv",
    ],
)

=== Di attack (extended) (di_extended) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_Di_ex.py out_anonymized/DALL_multi.json in/PWSCUP2025_Pre_Data_for_Attack/A22.csv --out-pred out_attack/inferred_membership1_22_ex_DALL.csv --out-conf out_attack/inferred_membership2_22_ex_DALL.csv
inferred was successfully saved.
inferred was successfully saved.


### Di attack (extended, DALLB model)

Repeats `attack/attack_Di_ex.py` with the Bi-variant anonymized booster `out_anonymized/DALL_multi_Bi.json`.  By keeping the same command structure, it generates Pred/Conf outputs (`..._ex_DALLB.csv`) aligned with the other Di runs for subsequent aggregation.

このセルでは `attack/attack_Di_ex.py` に Bi 版匿名化モデル `out_anonymized/DALL_multi_Bi.json` を指定し、Pred/Conf の推定結果を `out_attack/inferred_membership1_{id:02d}_ex_DALLB.csv` と `out_attack/inferred_membership2_{id:02d}_ex_DALLB.csv` に保存します。他の Di 系出力と揃った形式で Bi モデルの影響を評価できます。


In [ ]:
# DALLB
run_command(
    key="di_extended",
    label="Di attack (extended)",
    category="di",
    description="Extended Di attack with prediction and confidence outputs.",
    template=[
        "python",
        "attack/attack_Di_ex.py",
        f"out_anonymized/DALL_multi_Bi.json",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        "--out-pred",
        "out_attack/inferred_membership1_{id:02d}_ex_DALLB.csv",
        "--out-conf",
        "out_attack/inferred_membership2_{id:02d}_ex_DALLB.csv",
    ],
)

=== Di attack (extended) (di_extended) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_Di_ex.py out_anonymized/DALL_multi_Bi.json in/PWSCUP2025_Pre_Data_for_Attack/A22.csv --out-pred out_attack/inferred_membership1_22_ex_DALLB.csv --out-conf out_attack/inferred_membership2_22_ex_DALLB.csv
inferred was successfully saved.
inferred was successfully saved.


## Combination Attacks


In [ ]:

# run_command(
#     key="combi_original",
#     label="Combination attack (original)",
#     category="combination",
#     description="Combine Ci and Di original outputs.",
#     template=[
#         "python",
#         "attack/attack_example_ex.py",
#         "--Ai_csv",
#         f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
#         "-o",
#         f"out_attack/Fij_{{id:02d}}.csv",
#         f"out_attack/{mode_anon}{{id:02d}}_inferred.csv",
#         "out_attack/inferred_membership1_{id:02d}_ex.csv",
#         "out_attack/inferred_membership2_{id:02d}_ex.csv",
#     ],
# )

### Combination attack (extended)

Feeds the greedy Ci result and Di Pred/Conf files into `attack/attack_example_ex.py`, enforcing a `-l 10000` cap so only the top 10k vote totals remain.  The command writes the aggregated membership decisions to `in/Fij_{id:02d}.csv`, which later stages treat as the combined inference for evaluation.

このセルでは `attack/attack_example_ex.py` に Ci greedy 出力と Di の Pred/Conf を入力し、票の合算後に上位 10,000 件だけを `in/Fij_{id:02d}.csv` に書き出します。これが後続評価で用いる統合推定結果になります。


In [ ]:

run_command(
    key="combi_extended",
    label="Combination attack (extended)",
    category="combination",
    description="Extended combination attack with limit 10000.",
    template=[
        "python",
        "attack/attack_example_ex.py",
        "--Ai_csv",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        "-o",
        "in/Fij_{id:02d}.csv",
        "-l",
        "10000",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_ex_greedy_k300_greedy.csv",
        "out_attack/inferred_membership1_{id:02d}_ex.csv",
        "out_attack/inferred_membership2_{id:02d}_ex.csv",
    ],
)


### Combination attack (extended, DALL inputs)

Repeats `attack/attack_example_ex.py` but supplies the DALL-specific Ci/Di artefacts (`..._hungarian_auto_k300.csv`, `..._ex_DALL.csv`).  The same 10k limit yields `in/Fij_{id:02d}_DALL.csv`, aligning anonymized-model combinations with the baseline format.

このセルでは DALL 用の Ci/Di 出力を `attack/attack_example_ex.py` に与え、同じく上位 10,000 件に精選した組み合わせ結果を `in/Fij_{id:02d}_DALL.csv` に保存します。匿名化モデルに対応した統合推定をベースラインと並列に管理します。


In [ ]:
# DALL
run_command(
    key="combi_extended",
    label="Combination attack (extended)",
    category="combination",
    description="Extended combination attack with limit 10000.",
    template=[
        "python",
        "attack/attack_example_ex.py",
        "--Ai_csv",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        "-o",
        "in/Fij_{id:02d}_DALL.csv",
        "-l",
        "10000",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_hungarian_auto_k300.csv",
        "out_attack/inferred_membership1_{id:02d}_ex_DALL.csv",
        "out_attack/inferred_membership2_{id:02d}_ex_DALL.csv",
    ],
)

=== Combination attack (extended) (combi_extended) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_example_ex.py --Ai_csv in/PWSCUP2025_Pre_Data_for_Attack/A22.csv -o in/Fij_22_DALL.csv -l 10000 out_attack/C22_inferred_hungarian_auto_k300.csv out_attack/inferred_membership1_22_ex_DALL.csv out_attack/inferred_membership2_22_ex_DALL.csv
[MixAttack] top-10000 selected (limit=10000)
[MixAttack] result: selected=10000/100000 (Ci sum=10000, Pred sum=89896, Conf sum=65845)
inferred was successfully saved as in/Fij_22_DALL.csv


### Combination attack (extended, DALLB inputs)

Combines the Bi-model Ci and Di outputs using the same `attack/attack_example_ex.py` command structure, producing `in/Fij_{id:02d}_DALLB.csv` after limiting to the strongest 10k votes.  This keeps all combination artefacts synchronised for comparison across model variants.

このセルでは Bi モデル由来の Ci/Di 出力をまとめて `attack/attack_example_ex.py` に渡し、上位 10,000 件の集計結果を `in/Fij_{id:02d}_DALLB.csv` に保存します。各モデル変種の組み合わを同一フォーマットで比較できるようにしています。


In [ ]:
# DALLB
run_command(
    key="combi_extended",
    label="Combination attack (extended)",
    category="combination",
    description="Extended combination attack with limit 10000.",
    template=[
        "python",
        "attack/attack_example_ex.py",
        "--Ai_csv",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        "-o",
        "in/Fij_{id:02d}_DALLB.csv",
        "-l",
        "10000",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_hungarian_auto_k300.csv",
        "out_attack/inferred_membership1_{id:02d}_ex_DALLB.csv",
        "out_attack/inferred_membership2_{id:02d}_ex_DALLB.csv",
    ],
)

=== Combination attack (extended) (combi_extended) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_example_ex.py --Ai_csv in/PWSCUP2025_Pre_Data_for_Attack/A22.csv -o in/Fij_22_DALLB.csv -l 10000 out_attack/C22_inferred_hungarian_auto_k300.csv out_attack/inferred_membership1_22_ex_DALLB.csv out_attack/inferred_membership2_22_ex_DALLB.csv
[MixAttack] top-10000 selected (limit=10000)
[MixAttack] result: selected=10000/100000 (Ci sum=10000, Pred sum=89910, Conf sum=65810)
inferred was successfully saved as in/Fij_22_DALLB.csv


## Scoring Strategies


In [ ]:

run_command(
    key="new_dici_scoring",
    label="New Di->Ci scoring",
    category="scoring",
    description="Rank Di candidates by Ci distance and prediction error (union mode).",
    template=[
        "python",
        "attack/new_attackDi_Ci.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_model}{{id:02d}}.json",
        "--pred-threshold",
        "0.5",
        "--conf-threshold",
        "0.25",
        "--mode",
        "intersection",
        "-k",
        "5",
        "--w-conf",
        "1.0",
        "--topn",
        "10000",
        "-o",
        "out_attack/Fij_new_{id:02d}.csv",
        "--out-rank",
        "out_attack/Fij_new_{id:02d}_rank.csv",
    ],
)


In [ ]:

run_command(
    key="new_dici_scoring_greedy",
    label="New Di->Ci scoring (greedy)",
    category="scoring",
    description="Greedy expansion variant of the new Di->Ci scoring attack.",
    template=[
        "python",
        "attack/new_attackDi_Ci_greedy.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_model}{{id:02d}}.json",
        "--pred-threshold",
        "0.5",
        "--conf-threshold",
        "0.25",
        "--mode",
        "intersection",
        "--w-conf",
        "1.0",
        "--topn",
        "10000",
        "-o",
        "out_attack/Fij_new_greedy_{id:02d}.csv",
        "--out-rank",
        "out_attack/Fij_new_greedy_{id:02d}_rank.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_greedy.csv",
    ],
)


In [ ]:

run_command(
    key="ci_di_independent",
    label="Independent Ci + Di scoring",
    category="scoring",
    description="Combine Ci distance and Di prediction error independently.",
    template=[
        "python",
        "attack/attack_Ci_Di_independent.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_model}{{id:02d}}.json",
        "--w-conf",
        "1.0",
        "--k-hint",
        "300",
        "--topn",
        "10000",
        "-o",
        "out_attack/Fij_independent_{id:02d}.csv",
        "--out-rank",
        "out_attack/Fij_independent_{id:02d}_rank.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_independent.csv",
    ],
)


In [ ]:

run_command(
    key="ci_hungarian",
    label="Ci attack (Hungarian)",
    category="ci",
    description="Hungarian assignment variant of the Ci attack (auto mode, k=300).",
    template=[
        "python",
        "attack/attack_Ci_hungarian.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        "-m",
        "auto",
        "-k",
        "300",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_hungarian_auto_k300.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_hungarian_auto_k300.csv",
    ],
)


=== Ci attack (Hungarian) (ci_hungarian) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_Ci_hungarian.py in/PWSCUP2025_Pre_Data_for_Attack/A22.csv in/PWSCUP2025_Pre_Data_for_Attack/C22_fix.csv -m auto -k 300 -o out_attack/C22_inferred_hungarian_auto_k300.csv --out-map out_attack/C22_matchmap_hungarian_auto_k300.csv
[AttackCiHungarian] matched=10000 of Ci=10000; selected Ai=10000/100000
inferred was successfully saved as out_attack/C22_inferred_hungarian_auto_k300.csv
match table was successfully saved as out_attack/C22_matchmap_hungarian_auto_k300.csv


In [7]:
run_command(
    key="distribution_weights",
    label="creation_weights",
    category="ci",
    description="Hungarian assignment with weight variant of the Ci attack (auto mode, k=300).",
    template=[
        "python",
        "attack/distribution_weight.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv", #baseline.csv 
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv", #target.csv
        "-o",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}_weights.json" #weights.json
    ],
)


=== creation_weights (distribution_weights) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/distribution_weight.py in/PWSCUP2025_Main_Data_for_Attack/AA01.csv in/PWSCUP2025_Main_Data_for_Attack/CC01.csv -o in/PWSCUP2025_Main_Data_for_Attack/CC01_weights.json


Wrote 18 weights to in/PWSCUP2025_Main_Data_for_Attack/CC01_weights.json
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/distribution_weight.py in/PWSCUP2025_Main_Data_for_Attack/AA02.csv in/PWSCUP2025_Main_Data_for_Attack/CC02.csv -o in/PWSCUP2025_Main_Data_for_Attack/CC02_weights.json
Wrote 18 weights to in/PWSCUP2025_Main_Data_for_Attack/CC02_weights.json
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/distribution_weight.py in/PWSCUP2025_Main_Data_for_Attack/AA03.csv in/PWSCUP2025_Main_Data_for_Attack/CC03.csv -o in/PWSCUP2025_Main_Data_for_Attack/CC03_weights.json
Wrote 18 weights to in/PWSCUP2025_Main_Data_for_Attack/CC03_weights.json
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/distribution_weight.py in/PWSCUP2025_Main_Data_for_Attack/AA04.csv in/PWSCUP2025_Main_Data_for_Attack/CC04.csv -o in/PWSCUP2025_Main_Data_for_Attack/CC04_weights.json
Wrote 18 weights to in/PWSCUP2025_Main_Data_for_Attack/CC04_weights.json
>> /home/kikuchih/

In [8]:

run_command(
    key="ci_hungarian_weighted",
    label="Ci attack (Hungarian-weighted)",
    category="ci",
    description="Hungarian assignment with weight variant of the Ci attack (auto mode, k=300).",
    template=[
        "python",
        "attack/attack_Ci_hungarian_weighted.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        "-m",
        "auto",
        "-k",
        "300",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_hungarian_auto_k300_weighted.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_hungarian_auto_k300_weighted.csv",
        "--weights-file",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}_weights.json"
    ],
)


=== Ci attack (Hungarian-weighted) (ci_hungarian_weighted) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_Ci_hungarian_weighted.py in/PWSCUP2025_Main_Data_for_Attack/AA01.csv in/PWSCUP2025_Main_Data_for_Attack/CC01.csv -m auto -k 300 -o out_attack/CC01_inferred_hungarian_auto_k300_weighted.csv --out-map out_attack/CC01_matchmap_hungarian_auto_k300_weighted.csv --weights-file in/PWSCUP2025_Main_Data_for_Attack/CC01_weights.json


Using column weights from in/PWSCUP2025_Main_Data_for_Attack/CC01_weights.json: {'GENDER': 0.7937628714327739, 'AGE': 0.0, 'RACE': 0.7820927723840341, 'ETHNICITY': 0.6832401686770618, 'encounter_count': 0.39688143571638723, 'num_procedures': 0.3991370010787486, 'num_medications': 0.6371481808375011, 'num_immunizations': 0.7046190055898794, 'num_allergies': 0.9251740708051384, 'num_devices': 0.7684613121506325, 'asthma_flag': 1.0, 'stroke_flag': 1.0, 'obesity_flag': 1.0, 'depression_flag': 1.0, 'mean_systolic_bp': 0.6368539766598019, 'mean_diastolic_bp': 0.6123369618515249, 'mean_bmi': 0.24830832597822883, 'mean_weight': 0.49514563106796117}
[AttackCiHungarian] matched=10000 of Ci=10000; selected Ai=10000/100000
inferred membership saved to out_attack/CC01_inferred_hungarian_auto_k300_weighted.csv
match table saved as out_attack/CC01_matchmap_hungarian_auto_k300_weighted.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_Ci_hungarian_weighted.py in/PWSCUP2025_Main

In [12]:
run_command(
    key="ci_hungarian_weighted_entropy",
    label="Ci attack (Hungarian-weighted-entropy)",
    category="ci",
    description="Hungarian assignment with entropy-weighted columns (auto mode, k=300).",
    template=[
        "python",
        "attack/attack_Ci_hungarian_weighted_entropy.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        "-m",
        "auto",
        "-k",
        "300",
        "-o",
        f"out_attack/{mode_anon}{{id:02d}}_inferred_hungarian_auto_k300_weighted_entropy.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_hungarian_auto_k300_weighted_entropy.csv",
        "--weights-file",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}_weights.json",
        "--alpha",
        "0.5",
    ],
)

=== Ci attack (Hungarian-weighted-entropy) (ci_hungarian_weighted_entropy) ===
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_Ci_hungarian_weighted_entropy.py in/PWSCUP2025_Main_Data_for_Attack/AA01.csv in/PWSCUP2025_Main_Data_for_Attack/CC01.csv -m auto -k 300 -o out_attack/CC01_inferred_hungarian_auto_k300_weighted_entropy.csv --out-map out_attack/CC01_matchmap_hungarian_auto_k300_weighted_entropy.csv --weights-file in/PWSCUP2025_Main_Data_for_Attack/CC01_weights.json --alpha 0.5


Using column weights from in/PWSCUP2025_Main_Data_for_Attack/CC01_weights.json
match map saved to out_attack/CC01_matchmap_hungarian_auto_k300_weighted_entropy.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_Ci_hungarian_weighted_entropy.py in/PWSCUP2025_Main_Data_for_Attack/AA02.csv in/PWSCUP2025_Main_Data_for_Attack/CC02.csv -m auto -k 300 -o out_attack/CC02_inferred_hungarian_auto_k300_weighted_entropy.csv --out-map out_attack/CC02_matchmap_hungarian_auto_k300_weighted_entropy.csv --weights-file in/PWSCUP2025_Main_Data_for_Attack/CC02_weights.json --alpha 0.5
Using column weights from in/PWSCUP2025_Main_Data_for_Attack/CC02_weights.json
match map saved to out_attack/CC02_matchmap_hungarian_auto_k300_weighted_entropy.csv
>> /home/kikuchih/miniconda3/envs/pwscup2025/bin/python attack/attack_Ci_hungarian_weighted_entropy.py in/PWSCUP2025_Main_Data_for_Attack/AA03.csv in/PWSCUP2025_Main_Data_for_Attack/CC03.csv -m auto -k 300 -o out_attack/CC03_inferred_hungari

In [ ]:

run_command(
    key="new_dici_hungarian",
    label="New Di->Ci scoring (Hungarian)",
    category="scoring",
    description="Apply Di->Ci scoring after Hungarian Ci matching.",
    template=[
        "python",
        "attack/attackDi_Ci_hungarian.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_model}{{id:02d}}.json",
        "--pred-threshold",
        "0.5",
        "--conf-threshold",
        "0.25",
        "--mode",
        "intersection",
        "--hung-mode",
        "auto",
        "-k",
        "300",
        "--w-conf",
        "1.0",
        "--topn",
        "10000",
        "-o",
        "out_attack/Fij_new_hung_{id:02d}.csv",
        "--out-rank",
        "out_attack/Fij_new_hung_{id:02d}_rank.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_hungarian_used.csv",
    ],
)


In [ ]:

run_command(
    key="allci_alldi_hungarian",
    label="AllCi + AllDi (Hungarian)",
    category="scoring",
    description="Aggregate all Ci and Di candidates with Hungarian scoring.",
    template=[
        "python",
        "attack/attack_allCi_allDi_hungarian.py",
        f"in/{mode_attack_data}/{mode_all}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_anon}{{id:02d}}.csv",
        f"in/{mode_attack_data}/{mode_model}{{id:02d}}.json",
        "--hung-mode",
        "auto",
        "-k",
        "300",
        "--w-dist",
        "1.0",
        "--w-conf",
        "1.0",
        "--topn",
        "10000",
        "-o",
        "out_attack/Fij_all_hungarian_{id:02d}.csv",
        "--out-rank",
        "out_attack/Fij_all_hungarian_{id:02d}_rank.csv",
        "--out-map",
        f"out_attack/{mode_anon}{{id:02d}}_matchmap_all_hungarian.csv",
    ],
)


## Pipeline Controls
- Call list_commands() to view registered steps.
- Use the toggles below to preview or rerun commands.


In [ ]:
RUN_PIPELINE = False  # set to True to execute the recorded sequence
DRY_RUN = True        # set to False to launch subprocesses
TARGET_TEAMS = None   # e.g., TEAM_22_ONLY for quick checks

if RUN_PIPELINE:
    run_pipeline(teams=TARGET_TEAMS, dry_run=DRY_RUN)


## Dataset Checklist
- Ensure the in/ directory exists at the project root.
- Place PWSCUP2025_Pre_Data_for_Attack archives under in/ and extract them to in/PWSCUP2025_Pre_Data_for_Attack/.
- Example layout: in/PWSCUP2025_Pre_Data_for_Attack/A01.csv


## Answer Generation
Example: python evaluation/gen_ans.py in/PWSCUP2025_Pre_Data_for_Attack/A22.csv in/B22_3.csv -o in/Z22.csv


## Evaluation
Example: python evaluation/check_ans.py out_attack/Fij_all_hungarian_22.csv in/Z22.csv
